# 03 EDA

ทำความเข้าใจข้อมูลก่อนตอบคำถามธุรกิจ (Q1-Q4) และก่อนทำโมเดล
ทุกหัวข้อจบด้วยข้อสังเกตสั้นๆ ว่าเห็นอะไร และเอาไปใช้ต่อที่ไหน

ข้อมูลที่ใช้
- marts จาก notebook 02 เป็นหลัก ใช้เฉพาะวัน normal
- fact table (`od`) เฉพาะงานที่ต้องดูระดับ OD-ชั่วโมง
- เทียบก่อน/หลัง COVID ด้วย ม.ค.-เม.ย. 2019 กับ ม.ค.-เม.ย. 2023 (ช่วงเดียวกับ Q1-Q4)

หัวข้อ
1. ภาพรวมและการกระจาย
2. รูปแบบตามเวลา
3. สถานี
4. Origin-Destination
5. ความสัมพันธ์ตามเวลา (สำหรับ M1)
6. สรุป EDA

In [1]:
import sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()
sys.path.append(str(ROOT / "src"))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from bart import viz
from bart.config import create_views, duck, fetch_row, fetch_value
from bart.perf import stage
from bart.stations import STATIONS

viz.notebook_mode()

con = duck()
create_views(con)

ช่วงเวลาที่ใช้เทียบก่อน/หลัง COVID ใช้ซ้ำทั้ง notebook

In [2]:
periods = {
    "Jan-Apr 2019": ("2019-01-01", "2019-04-30"),
    "Jan-Apr 2023": ("2023-01-01", "2023-04-30"),
}


def period_case(column="date"):
    lines = ["CASE"]
    for label, (start, end) in periods.items():
        lines.append(f"    WHEN {column} BETWEEN DATE '{start}' AND DATE '{end}' THEN '{label}'")
    lines.append("END")
    return "\n".join(lines)


def bucket_label(bucket):
    if bucket < 0:
        return "0"
    if bucket == 0:
        return "1"
    if bucket == 1:
        return "2"
    low = 2 ** (bucket - 1) + 1
    high = 2 ** bucket
    return f"{low:,}-{high:,}"


print(period_case())

CASE
    WHEN date BETWEEN DATE '2019-01-01' AND DATE '2019-04-30' THEN 'Jan-Apr 2019'
    WHEN date BETWEEN DATE '2023-01-01' AND DATE '2023-04-30' THEN 'Jan-Apr 2023'
END


## 1. ภาพรวมและการกระจาย

### 1.1 ภาพรวมรายปี

ใช้เฉพาะทริปที่ทั้งสถานีต้นทางและปลายทางเปิดแล้ว (ตัดทริปหลงก่อนวันเปิดเหมือน notebook 02)

คู่ OD ที่เป็นไปได้ = จำนวนสถานีที่เปิดในวันนั้น ยกกำลังสอง (นับคู่ที่เข้าออกสถานีเดียวกันด้วย)
active_pct คือค่าเฉลี่ยรายวันว่าคู่ OD ที่เป็นไปได้ มีการเดินทางจริงกี่ % ในหนึ่งวัน

In [3]:
overview_query = """
    WITH valid AS (
        SELECT t.date, t.origin, t.dest, t.trips
        FROM od AS t
        JOIN stations AS o ON o.station = t.origin
        JOIN stations AS d ON d.station = t.dest
        JOIN calendar AS c ON c.date = t.date
        WHERE t.date >= o.opening_date
          AND t.date >= d.opening_date
          AND c.day_status = 'normal'
    ),
    pairs AS (
        SELECT DISTINCT date, origin, dest
        FROM valid
    ),
    daily_pairs AS (
        SELECT date, count(*) AS active_pairs
        FROM pairs
        GROUP BY date
    ),
    daily_possible AS (
        SELECT c.date, count(*) * count(*) AS possible_pairs
        FROM calendar AS c
        JOIN stations AS s ON s.opening_date <= c.date
        WHERE c.day_status = 'normal'
        GROUP BY c.date
    ),
    daily AS (
        SELECT
            p.date,
            p.active_pairs,
            q.possible_pairs,
            p.active_pairs / q.possible_pairs AS active_share
        FROM daily_pairs AS p
        JOIN daily_possible AS q ON q.date = p.date
    ),
    yearly AS (
        SELECT
            year(date)  AS year,
            count(*)    AS rows,
            sum(trips)  AS trips
        FROM valid
        GROUP BY year(date)
    )
    SELECT
        y.year,
        y.rows,
        y.trips,
        count(*)                                  AS normal_days,
        round(y.trips / count(*))                 AS trips_per_day,
        max(d.possible_pairs)                     AS possible_pairs,
        round(avg(d.active_pairs))                AS active_pairs_per_day,
        round(100 * avg(d.active_share), 1)       AS active_pct
    FROM yearly AS y
    JOIN daily AS d ON year(d.date) = y.year
    GROUP BY y.year, y.rows, y.trips
    ORDER BY y.year
"""

with stage("03.overview"):
    overview = con.sql(overview_query).df()

overview

[perf] 03.overview: 2.04s | 17.8 cores | peak RAM 9.829 GB | CPU 92.5%


,year,rows,trips,normal_days,trips_per_day,possible_pairs,active_pairs_per_day,active_pct
0,2011,9421009,106364656.0,361,294639.0,1936,1914.0,99.5
1,2012,9769509,115407813.0,365,316186.0,1936,1928.0,99.6
2,2013,9591284,116844427.0,356,328215.0,1936,1928.0,99.6
3,2014,9889275,124011318.0,365,339757.0,2025,1937.0,99.6
4,2015,10072503,127723465.0,365,349927.0,2025,2012.0,99.3
5,2016,9971540,128176173.0,366,350208.0,2025,2007.0,99.1
6,2017,9972039,122849865.0,365,336575.0,2116,2073.0,99.0
7,2018,10227572,120494127.0,365,330121.0,2304,2206.0,99.0
8,2019,10095473,118500199.0,364,325550.0,2304,2274.0,98.7
9,2020,5811792,31426070.0,362,86812.0,2500,2147.0,89.0


### 1.2 ความเบาบางของข้อมูล OD-ชั่วโมง

แต่ละแถวของ fact table คือ คู่ OD × ชั่วโมง ที่มีอย่างน้อย 1 ทริป ดูว่าแถวส่วนใหญ่มีกี่ทริป

In [4]:
sparsity_query = f"""
    WITH rows_in_period AS (
        SELECT
            {period_case("o.date")} AS period,
            o.trips
        FROM od AS o
        JOIN calendar AS c ON c.date = o.date
        WHERE c.day_status = 'normal'
    )
    SELECT
        period,
        count(*)                                                   AS rows,
        round(100 * avg(CASE WHEN trips = 1 THEN 1 ELSE 0 END), 1) AS pct_rows_1_trip,
        round(100 * avg(CASE WHEN trips <= 2 THEN 1 ELSE 0 END), 1) AS pct_rows_1_2_trips,
        quantile_disc(trips, 0.5)                                  AS p50,
        quantile_disc(trips, 0.9)                                  AS p90,
        quantile_disc(trips, 0.99)                                 AS p99,
        max(trips)                                                 AS max
    FROM rows_in_period
    WHERE period IS NOT NULL
    GROUP BY period
    ORDER BY period
"""

with stage("03.sparsity"):
    sparsity = con.sql(sparsity_query).df()

sparsity

[perf] 03.sparsity: 0.39s | 5.97 cores | peak RAM 9.739 GB | CPU 50.4%


,period,rows,pct_rows_1_trip,pct_rows_1_2_trips,p50,p90,p99,max
0,Jan-Apr 2019,3301290,27.3,42.9,3,24,156,945
1,Jan-Apr 2023,2765277,36.0,53.9,2,12,53,320


แบ่งแถวเป็นกลุ่มตามจำนวนทริป (1, 2, 3-4, 5-8, ...) แล้วดูว่าแต่ละกลุ่มมีสัดส่วนแถวเท่าไร และสัดส่วนทริปเท่าไร ใช้ปี 2019 ทั้งปี

In [5]:
trip_buckets_query = """
    SELECT
        ceil(log2(o.trips))::INTEGER AS bucket,
        count(*)                     AS rows,
        sum(o.trips)                 AS trips
    FROM od AS o
    JOIN calendar AS c ON c.date = o.date
    WHERE c.day_status = 'normal' AND c.year = 2019
    GROUP BY bucket
    ORDER BY bucket
"""

with stage("03.trip_buckets"):
    trip_buckets = con.sql(trip_buckets_query).df()

labels = []
for bucket in trip_buckets["bucket"]:
    labels.append(bucket_label(bucket))
trip_buckets["trips_per_row"] = labels
trip_buckets["pct_rows"] = 100 * trip_buckets["rows"] / trip_buckets["rows"].sum()
trip_buckets["pct_trips"] = 100 * trip_buckets["trips"] / trip_buckets["trips"].sum()

trip_buckets.round(2)

[perf] 03.trip_buckets: 0.06s | 7.77 cores | peak RAM 9.487 GB


,bucket,rows,trips,trips_per_row,pct_rows,pct_trips
0,0,2724216,2724216.0,1,26.98,2.30
1,1,1562923,3125846.0,2,15.48,2.64
2,2,1727462,5896907.0,3-4,17.11,4.98
3,3,1525444,9474868.0,5-8,15.11,8.00
4,4,1098322,12934225.0,9-16,10.88,10.91
5,5,696613,15984847.0,17-32,6.90,13.49
6,6,413006,18671550.0,33-64,4.09,15.76
7,7,215201,19143434.0,65-128,2.13,16.15
8,8,95587,16771245.0,129-256,0.95,14.15
9,9,32564,11083829.0,257-512,0.32,9.35


In [6]:
rows_bars = go.Bar(
    name="Share of rows",
    x=trip_buckets["trips_per_row"],
    y=trip_buckets["pct_rows"],
    marker_color=viz.CONTEXT,
)
trips_bars = go.Bar(
    name="Share of trips",
    x=trip_buckets["trips_per_row"],
    y=trip_buckets["pct_trips"],
    marker_color=viz.ACCENT,
)

fig = go.Figure([rows_bars, trips_bars])
fig.update_traces(hovertemplate="%{x} trips: %{y:.1f}%<extra>%{fullData.name}</extra>")
fig.update_layout(barmode="group", hovermode="closest")
fig.update_xaxes(title_text="trips in one OD-hour row")
fig.update_yaxes(title_text="% of total", ticksuffix="%")

title = "Most OD-hour rows carry only a few trips"
subtitle = "OD-hour rows in 2019 (normal days) grouped by trip count. Grey = share of rows, blue = share of trips"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_od_hour_sparsity")
fig

### 1.3 entries ต่อสถานี-ชั่วโมง

ใช้ station_hour ที่เติมศูนย์แล้ว ดูว่ามีชั่วโมงที่เป็นศูนย์เท่าไร แยกช่วงตี 1-4 (แทบไม่มีรถวิ่ง) ออกจากช่วงอื่น

In [7]:
zero_query = f"""
    WITH hours_in_period AS (
        SELECT
            {period_case("h.date")} AS period,
            h.hour,
            h.entries
        FROM station_hour AS h
        JOIN calendar AS c ON c.date = h.date
        WHERE c.day_status = 'normal'
    )
    SELECT
        period,
        CASE
            WHEN hour BETWEEN 1 AND 4 THEN '01-04 (night)'
            ELSE 'other hours'
        END AS hours,
        count(*)                                                     AS station_hours,
        round(100 * avg(CASE WHEN entries = 0 THEN 1 ELSE 0 END), 1) AS pct_zero,
        round(avg(entries), 1)                                       AS mean_entries
    FROM hours_in_period
    WHERE period IS NOT NULL
    GROUP BY period, hours
    ORDER BY period, hours
"""

zero_share = con.sql(zero_query).df()
zero_share

,period,hours,station_hours,pct_zero,mean_entries
0,Jan-Apr 2019,01-04 (night),22848,69.2,3.7
1,Jan-Apr 2019,other hours,114240,2.8,339.3
2,Jan-Apr 2023,01-04 (night),24000,70.8,1.4
3,Jan-Apr 2023,other hours,120000,2.6,125.9


In [8]:
entry_buckets_query = f"""
    WITH hours_in_period AS (
        SELECT
            {period_case("h.date")} AS period,
            h.entries
        FROM station_hour AS h
        JOIN calendar AS c ON c.date = h.date
        WHERE c.day_status = 'normal'
    )
    SELECT
        period,
        CASE
            WHEN entries = 0 THEN -1
            ELSE ceil(log2(entries))::INTEGER
        END AS bucket,
        count(*) AS station_hours
    FROM hours_in_period
    WHERE period IS NOT NULL
    GROUP BY period, bucket
    ORDER BY period, bucket
"""

entry_buckets = con.sql(entry_buckets_query).df()

labels = []
for bucket in entry_buckets["bucket"]:
    labels.append(bucket_label(bucket))
entry_buckets["entries"] = labels

period_totals = entry_buckets.groupby("period")["station_hours"].transform("sum")
entry_buckets["pct"] = 100 * entry_buckets["station_hours"] / period_totals

entry_buckets.pivot(index="bucket", columns="period", values="pct").round(1)

period,Jan-Apr 2019,Jan-Apr 2023
bucket,,
-1,13.8,13.9
0,1.7,3.0
1,1.2,1.8
2,1.5,2.4
3,2.3,3.7
4,3.7,6.4
5,6.1,11.2
6,11.1,16.0
7,14.7,18.1


In [9]:
fig = go.Figure()

colors = {"Jan-Apr 2019": viz.CONTEXT, "Jan-Apr 2023": viz.CONTRAST}
for period, color in colors.items():
    rows = entry_buckets[entry_buckets["period"] == period]
    fig.add_trace(go.Bar(name=period, x=rows["entries"], y=rows["pct"], marker_color=color))

fig.update_traces(hovertemplate="%{x} entries: %{y:.1f}%<extra>%{fullData.name}</extra>")
fig.update_layout(barmode="group", hovermode="closest")
fig.update_layout(margin=dict(b=120, r=70))
fig.update_xaxes(title_text="entries in one station-hour", tickangle=-45)
fig.update_yaxes(title_text="% of station-hours", ticksuffix="%")

busy_share = {}
for period in colors:
    rows = entry_buckets[entry_buckets["period"] == period]
    busy_rows = rows[rows["bucket"] >= 10]
    busy_share[period] = busy_rows["pct"].sum()

before = busy_share["Jan-Apr 2019"]
after = busy_share["Jan-Apr 2023"]
title = f"Busy station-hours (more than 512 entries) fell from {before:.0f}% to {after:.0f}%"
subtitle = "Entries per station-hour on normal days, Jan-Apr 2019 vs Jan-Apr 2023. The share of zero hours barely changed"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_entries_distribution", height=560)
fig

### 1.4 mean กับ variance

ถ้าข้อมูลเป็น Poisson ค่า variance จะเท่ากับ mean
ดูจากกลุ่ม สถานี × ชั่วโมง × ประเภทวัน ปี 2019 ว่า variance โตเร็วกว่า mean แค่ไหน

ถ้า variance ∝ mean^p ค่า p คือ power ของ Tweedie ที่เหมาะกับข้อมูล (p = 1 คือ Poisson, p = 2 คือ Gamma)

In [10]:
dispersion_query = """
    SELECT
        h.station::VARCHAR AS station,
        h.hour,
        c.day_type,
        avg(h.entries)      AS mean,
        var_samp(h.entries) AS variance
    FROM station_hour AS h
    JOIN calendar AS c ON c.date = h.date
    WHERE c.day_status = 'normal' AND c.year = 2019
    GROUP BY h.station, h.hour, c.day_type
    HAVING avg(h.entries) >= 1
"""

con.execute(f"CREATE OR REPLACE TEMP TABLE dispersion AS {dispersion_query}")
dispersion = con.sql("SELECT * FROM dispersion").df()

power, groups = fetch_row(con, """
    SELECT regr_slope(ln(variance), ln(mean)), count(*)
    FROM dispersion
    WHERE variance > 0
""")
overdispersed = fetch_value(con, "SELECT avg(CASE WHEN variance > mean THEN 1 ELSE 0 END) FROM dispersion")

print(f"groups (station x hour x day type): {groups:,}")
print(f"share of groups with variance > mean: {overdispersed:.1%}")
print(f"estimated power p (variance ~ mean^p): {power:.2f}")

groups (station x hour x day type): 2,006
share of groups with variance > mean: 99.7%
estimated power p (variance ~ mean^p): 1.37


In [11]:
low = dispersion["mean"].min()
high = dispersion["mean"].max()

points = go.Scatter(
    x=dispersion["mean"],
    y=dispersion["variance"],
    mode="markers",
    name="station x hour x day type",
    marker=dict(color=viz.ACCENT, size=5, opacity=0.4),
    customdata=dispersion[["station", "hour", "day_type"]],
    hovertemplate="%{customdata[0]} hour %{customdata[1]} (%{customdata[2]})<br>mean %{x:,.1f}, variance %{y:,.0f}<extra></extra>",
)
poisson_line = go.Scatter(
    x=[low, high],
    y=[low, high],
    mode="lines",
    name="variance = mean (Poisson)",
    line=dict(color=viz.MUTED, width=2, dash="dot"),
    hoverinfo="skip",
)

fig = go.Figure([points, poisson_line])
fig.update_layout(hovermode="closest")
fig.update_xaxes(type="log", title_text="mean entries")
fig.update_yaxes(type="log", title_text="variance of entries")

title = f"Variance grows like mean^{power:.1f}, far above Poisson"
subtitle = "Each dot is one station, hour and day type in 2019 (normal days). Log scales"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_mean_variance")
fig

### ข้อสังเกต (ส่วนที่ 1)

- ระดับวันข้อมูลหนาแน่น ในหนึ่งวันปกติคู่ OD ที่เป็นไปได้มีการเดินทางจริง ~99% ก่อน COVID และ ~95% ในปี 2022-2023
- แต่ระดับชั่วโมงเบาบางมาก ปี 2019 แถว OD-ชั่วโมง 27% มีแค่ 1 ทริป แถวพวกนี้รวมกันเป็นแค่ 2.3% ของทริปทั้งหมด ทริปส่วนใหญ่ (~60%) อยู่ในแถวที่มี 17-256 ทริป
- หลัง COVID ยิ่งเบาบางขึ้น แถวที่มี 1 ทริปเพิ่มจาก 27% เป็น 36% ค่า P99 ลดจาก 156 เหลือ 53 ทริป
- station-hour ช่วงตี 1-4 เป็นศูนย์ ~70% ส่วนชั่วโมงอื่นเป็นศูนย์แค่ ~3% ทั้งก่อนและหลัง COVID ศูนย์ส่วนใหญ่จึงเป็นศูนย์เพราะไม่มีรถวิ่ง ไม่ได้เกิดแบบสุ่ม
- สิ่งที่เปลี่ยนหลัง COVID คือชั่วโมงที่คนเยอะหายไป entries เฉลี่ยของชั่วโมงที่มีรถวิ่งลดจาก 339 เหลือ 126 ต่อสถานี-ชั่วโมง (เหลือ ~37%)
- variance มากกว่า mean ใน 99.7% ของกลุ่ม (overdispersion) Poisson จึงไม่พอ ค่า p ≈ 1.4 อยู่ระหว่าง Poisson (1) กับ Gamma (2) ตรงกับช่วงของ Tweedie ที่รองรับค่าศูนย์ได้ด้วย
- ข้อควรระวัง: variance ในแต่ละกลุ่มรวมการเปลี่ยนแปลงตลอดทั้งปีไว้ด้วย ค่า p จึงเป็นแค่จุดเริ่มต้น

ใช้ต่อที่ไหน
- M1: ใช้ objective Tweedie ให้ Optuna จูน variance power ในช่วง 1.1-1.9 (เริ่มที่ ~1.4)
- M1: ใช้ WAPE เป็นตัวชี้วัดหลัก เพราะมีศูนย์ ~14% ทำให้ MAPE หารด้วยศูนย์ไม่ได้ และชั่วโมงที่คนน้อยทำให้ % error สูงเกินจริง และรายงานผลแยกช่วงกลางคืน
- Q1: ระดับผู้โดยสารต่อชั่วโมงลดเหลือ ~37% เป็นตัวเลขตั้งต้นของการวิเคราะห์การฟื้นตัว
- Q3: ชั่วโมงที่คนเยอะหายไปมาก น่าดูว่าคอขวดใน Transbay Tube ยังมีอยู่ไหม

## 2. รูปแบบตามเวลา

### 2.1 trend รายเดือน

ค่าเฉลี่ยทริปต่อวัน แยกวันทำงานกับวันหยุด (เสาร์-อาทิตย์และวันหยุดราชการ) ใช้เฉพาะวัน normal
ไม่นับ พ.ค. 2023 เพราะมีข้อมูลแค่ 2 วัน

In [12]:
monthly_query = """
    SELECT
        month,
        day_type,
        avg(day_trips) AS trips_per_day,
        count(*)       AS days
    FROM calendar
    WHERE day_status = 'normal'
      AND month < DATE '2023-05-01'
    GROUP BY month, day_type
    ORDER BY month, day_type
"""

with stage("03.monthly_trend"):
    monthly = con.sql(monthly_query).df()

workday_2019 = fetch_value(con, """
    SELECT avg(day_trips) FROM calendar
    WHERE day_status = 'normal' AND is_workday AND year = 2019
""")
workday_2023 = fetch_value(con, """
    SELECT avg(day_trips) FROM calendar
    WHERE day_status = 'normal' AND is_workday AND date BETWEEN DATE '2023-01-01' AND DATE '2023-04-30'
""")
recovery_pct = 100 * workday_2023 / workday_2019

print(f"workday trips per day, 2019         : {workday_2019:,.0f}")
print(f"workday trips per day, Jan-Apr 2023 : {workday_2023:,.0f} ({recovery_pct:.0f}% of 2019)")

[perf] 03.monthly_trend: 0.01s | 1.97 cores | peak RAM 0.468 GB
workday trips per day, 2019         : 409,262
workday trips per day, Jan-Apr 2023 : 151,896 (37% of 2019)


In [13]:
line_styles = {
    "workday": dict(color=viz.ACCENT, width=2),
    "weekend_holiday": dict(color=viz.CONTRAST, width=2),
}
line_names = {"workday": "Workday", "weekend_holiday": "Weekend / holiday"}

fig = go.Figure()
for day_type, style in line_styles.items():
    rows = monthly[monthly["day_type"] == day_type]
    fig.add_trace(go.Scatter(
        x=rows["month"],
        y=rows["trips_per_day"],
        mode="lines",
        name=line_names[day_type],
        line=style,
        hovertemplate="%{x|%b %Y}: %{y:,.0f} trips per day<extra>%{fullData.name}</extra>",
    ))

viz.vline(fig, "2020-03-09", "COVID break", y=1.02)
fig.update_yaxes(tickformat="~s", title_text="average trips per day", rangemode="tozero")

title = f"Workday ridership in early 2023 is {recovery_pct:.0f}% of 2019"
subtitle = "Average trips per day by month, normal days only. Weekend and holiday riders recovered faster than workday riders"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_monthly_trend")
fig

In [14]:
weekend_2019 = fetch_value(con, """
    SELECT avg(day_trips) FROM calendar
    WHERE day_status = 'normal' AND NOT is_workday AND year = 2019
""")
weekend_2023 = fetch_value(con, """
    SELECT avg(day_trips) FROM calendar
    WHERE day_status = 'normal' AND NOT is_workday AND date BETWEEN DATE '2023-01-01' AND DATE '2023-04-30'
""")

print(f"weekend/holiday trips per day, 2019         : {weekend_2019:,.0f}")
print(f"weekend/holiday trips per day, Jan-Apr 2023 : {weekend_2023:,.0f} ({100 * weekend_2023 / weekend_2019:.0f}% of 2019)")

weekend/holiday trips per day, 2019         : 137,198
weekend/holiday trips per day, Jan-Apr 2023 : 70,569 (51% of 2019)


### 2.2 รูปทรงรายชั่วโมง

ค่าเฉลี่ยทริปในแต่ละชั่วโมง หารด้วยทริปทั้งวัน จะได้สัดส่วนของวัน ทำให้เทียบรูปทรงได้ แม้ระดับจะต่างกันมาก

In [15]:
hourly_query = f"""
    WITH hours_in_period AS (
        SELECT
            {period_case("s.date")} AS period,
            c.day_type,
            s.hour,
            s.trips
        FROM system_hour AS s
        JOIN calendar AS c ON c.date = s.date
        WHERE c.day_status = 'normal'
    ),
    hourly AS (
        SELECT period, day_type, hour, avg(trips) AS trips
        FROM hours_in_period
        WHERE period IS NOT NULL
        GROUP BY period, day_type, hour
    )
    SELECT
        period,
        day_type,
        hour,
        trips,
        100 * trips / sum(trips) OVER (PARTITION BY period, day_type) AS pct_of_day
    FROM hourly
    ORDER BY period, day_type, hour
"""

hourly = con.sql(hourly_query).df()
hourly.pivot(index="hour", columns=["period", "day_type"], values="pct_of_day").round(1)

period      Jan-Apr 2019            Jan-Apr 2023        
day_type weekend_holiday workday weekend_holiday workday
hour                                                    
0                    2.2     0.4             1.7     0.5
1                    0.6     0.1             0.5     0.1
2                    0.1     0.0             0.1     0.0
3                    0.1     0.0             0.0     0.0
4                    0.0     0.1             0.0     0.0
5                    0.0     1.1             0.1     1.2
6                    0.7     3.7             0.8     3.9
7                    1.7     8.1             2.0     7.9
8                    3.7    12.4             4.0    11.1
9                    5.0     9.5             5.0     7.7
10                   5.8     4.5             5.9     4.4
11                   6.7     2.9             7.0     3.5
12                   7.2     2.7             7.6     3.5
13                   7.2     2.8             7.4     3.7
14                   7.0     3.3             7.4     4.4
15                   7.3     4.6             7.6     5.9
16                   8.2     7.5             8.3     8.6
17                   8.6    12.1             8.6    11.8
18                   7.6    10.9             7.5     9.1
19                   5.7     5.6             5.8     4.9
20                   4.7     2.9             4.2     2.7
21                   3.9     2.0             3.6     1.9
22                   3.3     1.7             2.9     1.8
23                   2.5     1.2             2.2     1.2

In [16]:
profile_styles = {
    ("Jan-Apr 2019", "workday"): dict(color=viz.CONTEXT, width=3),
    ("Jan-Apr 2023", "workday"): dict(color=viz.ACCENT, width=3),
    ("Jan-Apr 2019", "weekend_holiday"): dict(color=viz.CONTEXT, width=2, dash="dot"),
    ("Jan-Apr 2023", "weekend_holiday"): dict(color=viz.CONTRAST, width=2, dash="dot"),
}

fig = go.Figure()
for (period, day_type), style in profile_styles.items():
    rows = hourly[(hourly["period"] == period) & (hourly["day_type"] == day_type)]
    fig.add_trace(go.Scatter(
        x=rows["hour"],
        y=rows["pct_of_day"],
        mode="lines",
        name=f"{line_names[day_type]}, {period}",
        line=style,
        hovertemplate="%{x}:00 - %{y:.1f}% of the day<extra>%{fullData.name}</extra>",
    ))

fig.update_xaxes(title_text="hour", dtick=2)
fig.update_yaxes(title_text="% of the day's trips", ticksuffix="%")

title = "2023 workdays: a smaller 8am peak, a busier midday, and the 5pm peak holds"
subtitle = "Share of the day's trips in each hour, normal days, Jan-Apr 2019 vs Jan-Apr 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_hourly_profile")
fig

สัดส่วนของช่วง peak เช้า (6:00-9:59) และเย็น (15:00-18:59) ในวันทำงาน

In [17]:
peak_query = """
    SELECT
        period,
        round(sum(pct_of_day) FILTER (WHERE hour BETWEEN 6 AND 9), 1)   AS am_peak_pct,
        round(sum(pct_of_day) FILTER (WHERE hour BETWEEN 15 AND 18), 1) AS pm_peak_pct,
        round(max(pct_of_day), 1)                                      AS busiest_hour_pct,
        round(max(pct_of_day) / avg(pct_of_day), 2)                    AS peak_to_average
    FROM hourly
    WHERE day_type = 'workday'
    GROUP BY period
    ORDER BY period
"""

con.sql(peak_query).df()

,period,am_peak_pct,pm_peak_pct,busiest_hour_pct,peak_to_average
0,Jan-Apr 2019,33.7,35.1,12.4,2.97
1,Jan-Apr 2023,30.6,35.5,11.8,2.83


### 2.3 วันในสัปดาห์ × ชั่วโมง

ตัดวันหยุดราชการออก ให้เหลือแค่ผลของวันในสัปดาห์
แต่ละช่องคือสัดส่วนของทริปทั้งสัปดาห์ (ผลรวมทุกช่อง = 100%)

In [18]:
week_query = f"""
    WITH hours_in_period AS (
        SELECT
            {period_case("s.date")} AS period,
            c.day_of_week,
            s.hour,
            s.trips
        FROM system_hour AS s
        JOIN calendar AS c ON c.date = s.date
        WHERE c.day_status = 'normal' AND NOT c.is_holiday
    ),
    cells AS (
        SELECT period, day_of_week, hour, avg(trips) AS trips
        FROM hours_in_period
        WHERE period IS NOT NULL
        GROUP BY period, day_of_week, hour
    )
    SELECT
        period,
        day_of_week,
        hour,
        100 * trips / sum(trips) OVER (PARTITION BY period) AS pct_of_week
    FROM cells
    ORDER BY period, day_of_week, hour
"""

week = con.sql(week_query).df()

In [19]:
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
highest = week["pct_of_week"].max()

fig = make_subplots(rows=1, cols=2, subplot_titles=list(periods), shared_yaxes=True, horizontal_spacing=0.04)

column = 1
for period in periods:
    grid = week[week["period"] == period].pivot(index="day_of_week", columns="hour", values="pct_of_week")
    fig.add_trace(
        go.Heatmap(
            z=grid.values,
            x=list(grid.columns),
            y=day_labels,
            colorscale=viz.SEQUENTIAL,
            zmin=0,
            zmax=highest,
            showscale=(column == 2),
            colorbar=dict(title="% of week", ticksuffix="%"),
            hovertemplate="%{y} %{x}:00 - %{z:.2f}% of the week<extra></extra>",
        ),
        row=1,
        col=column,
    )
    column += 1

fig.update_yaxes(autorange="reversed")
fig.update_xaxes(dtick=3)
fig.update_layout(hovermode="closest")

title = "The week looks the same shape, but Monday and Friday peaks faded in 2023"
subtitle = "Share of the week's trips by day and hour, normal non-holiday days"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_week_heatmap", height=480)
fig

ทริปต่อวันของแต่ละวันในสัปดาห์ เทียบกับค่าเฉลี่ยของ อังคาร-พฤหัส ในช่วงเดียวกัน

In [20]:
weekday_query = f"""
    WITH days_in_period AS (
        SELECT
            {period_case("date")} AS period,
            day_of_week,
            day_name,
            day_trips
        FROM calendar
        WHERE day_status = 'normal' AND NOT is_holiday
    ),
    by_day AS (
        SELECT period, day_of_week, day_name, avg(day_trips) AS trips_per_day
        FROM days_in_period
        WHERE period IS NOT NULL
        GROUP BY period, day_of_week, day_name
    )
    SELECT
        period,
        day_of_week,
        day_name,
        round(trips_per_day) AS trips_per_day,
        round(100 * trips_per_day / avg(trips_per_day) FILTER (WHERE day_of_week BETWEEN 2 AND 4)
              OVER (PARTITION BY period)) AS index_vs_tue_thu
    FROM by_day
    ORDER BY period, day_of_week
"""

weekdays = con.sql(weekday_query).df()
weekdays.pivot(index=["day_of_week", "day_name"], columns="period", values="index_vs_tue_thu")

,period,Jan-Apr 2019,Jan-Apr 2023
day_of_week,day_name,,
1,Monday,95.0,85.0
2,Tuesday,101.0,99.0
3,Wednesday,99.0,100.0
4,Thursday,100.0,101.0
5,Friday,92.0,83.0
6,Saturday,38.0,52.0
7,Sunday,25.0,37.0


### 2.4 ฤดูกาลรายเดือน

index ของแต่ละเดือน = ทริปเฉลี่ยต่อวันทำงานของเดือนนั้น / ทริปเฉลี่ยต่อวันทำงานทั้งปี × 100
ใช้ปี 2012-2019 (ปีเต็มก่อน COVID) เส้นคือค่าเฉลี่ย ส่วนแถบคือช่วง min-max ของ 8 ปี

In [21]:
season_query = """
    WITH by_month AS (
        SELECT year, month(date) AS month_of_year, avg(day_trips) AS trips_per_day
        FROM calendar
        WHERE day_status = 'normal' AND is_workday AND year BETWEEN 2012 AND 2019
        GROUP BY year, month(date)
    ),
    indexed AS (
        SELECT
            year,
            month_of_year,
            100 * trips_per_day / avg(trips_per_day) OVER (PARTITION BY year) AS idx
        FROM by_month
    )
    SELECT
        month_of_year,
        avg(idx) AS mean_index,
        min(idx) AS min_index,
        max(idx) AS max_index
    FROM indexed
    GROUP BY month_of_year
    ORDER BY month_of_year
"""

season = con.sql(season_query).df()
season.round(1)

,month_of_year,mean_index,min_index,max_index
0,1,95.9,93.1,98.3
1,2,100.3,97.4,103.9
2,3,99.7,97.1,101.1
3,4,100.5,98.6,102.0
4,5,100.7,99.3,101.8
5,6,101.1,98.5,102.6
6,7,99.8,98.9,101.4
7,8,101.2,99.4,102.4
8,9,104.6,101.6,106.3
9,10,103.8,100.1,109.3


In [22]:
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

band_top = go.Scatter(
    x=month_labels,
    y=season["max_index"],
    mode="lines",
    line=dict(width=0),
    showlegend=False,
    hoverinfo="skip",
)
band_bottom = go.Scatter(
    x=month_labels,
    y=season["min_index"],
    mode="lines",
    line=dict(width=0),
    fill="tonexty",
    fillcolor="rgba(42, 120, 214, 0.15)",
    name="min-max of 2012-2019",
    hoverinfo="skip",
)
mean_line = go.Scatter(
    x=month_labels,
    y=season["mean_index"],
    mode="lines+markers",
    name="average",
    line=dict(color=viz.ACCENT, width=2),
    marker=dict(size=8),
    hovertemplate="%{x}: %{y:.1f}<extra></extra>",
)

fig = go.Figure([band_top, band_bottom, mean_line])
fig.add_hline(y=100, line=dict(color=viz.MUTED, width=1, dash="dot"))
fig.update_layout(hovermode="closest")
fig.update_yaxes(title_text="index (year average = 100)")

lowest = season.loc[season["mean_index"].idxmin()]
highest_month = season.loc[season["mean_index"].idxmax()]
low_label = month_labels[int(lowest["month_of_year"]) - 1]
high_label = month_labels[int(highest_month["month_of_year"]) - 1]

title = f"Seasonality is mild: {low_label} is the low ({lowest['mean_index']:.0f}), {high_label} the high ({highest_month['mean_index']:.0f})"
subtitle = "Workday trips per day by month, indexed to each year's average, 2012-2019"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_seasonality")
fig

### 2.5 ผลของวันหยุด

ดูตารางใน notebook 02 (หัวข้อวันหยุด) Thanksgiving และ Christmas ผู้โดยสารเหลือ ~17-19% ของวันปกติ
Memorial Day, Labor Day, New Year's Day เหลือ ~30-35% ส่วน Columbus Day และ Veterans Day แทบไม่ต่างจากวันปกติ

### 2.6 วันพีคผิดปกติ

เทียบทริปของแต่ละวันกับค่ามัธยฐานของวันเดียวกันในสัปดาห์ ช่วง ±4 สัปดาห์ (ไม่นับวันหยุด)
ตัดช่วง ก.พ.-มิ.ย. 2020 ออก เพราะผู้โดยสารลดลงเร็วมากในช่วง COVID ทำให้ค่ามัธยฐานใช้เทียบไม่ได้

ใส่ชื่อเหตุการณ์เฉพาะที่ยืนยันได้จากวันที่ ถ้าไม่แน่ใจเขียนว่า unknown

In [23]:
def event_name(day):
    if day.month == 6 and day.isoweekday() == 7 and day.day >= 24:
        return "SF Pride parade (last Sunday of June)"

    known_events = {
        "2012-10-31": "Giants World Series parade",
        "2014-10-31": "Giants World Series parade",
        "2015-06-19": "Warriors championship parade",
        "2017-06-15": "Warriors championship parade",
        "2018-06-12": "Warriors championship parade",
        "2022-06-20": "Warriors championship parade",
        "2016-02-05": "Super Bowl 50 week, Super Bowl City in SF",
        "2016-02-06": "Super Bowl 50 week, Super Bowl City in SF",
        "2017-01-21": "Women's March",
    }
    key = day.strftime("%Y-%m-%d")
    if key in known_events:
        return known_events[key]
    return "unknown"


surge_query = """
    CREATE OR REPLACE TEMP TABLE surge AS
    WITH normal_days AS (
        SELECT date, day_name, day_trips, is_holiday, is_workday
        FROM calendar
        WHERE day_status = 'normal'
    )
    SELECT
        d.date,
        d.day_name,
        d.is_workday,
        d.day_trips,
        median(n.day_trips)                 AS usual_trips,
        d.day_trips / median(n.day_trips)   AS ratio
    FROM normal_days AS d
    JOIN normal_days AS n
        ON isodow(n.date) = isodow(d.date)
       AND n.date BETWEEN d.date - 28 AND d.date + 28
       AND n.date <> d.date
       AND NOT n.is_holiday
    WHERE d.date NOT BETWEEN DATE '2020-02-01' AND DATE '2020-06-30'
    GROUP BY d.date, d.day_name, d.is_workday, d.day_trips
"""

with stage("03.surge_days"):
    con.execute(surge_query)


def top_surges(where):
    table = con.sql(f"""
        SELECT date, day_name, day_trips, round(usual_trips) AS usual_trips, round(ratio, 2) AS ratio
        FROM surge
        WHERE {where}
        ORDER BY ratio DESC
        LIMIT 10
    """).df()

    events = []
    for day in table["date"]:
        events.append(event_name(day))
    table["event"] = events
    return table


top_surges("true")

[perf] 03.surge_days: 0.01s | 2.94 cores | peak RAM 0.479 GB


,date,day_name,day_trips,usual_trips,ratio,event
0,2015-06-28,Sunday,333073.0,152885.0,2.18,SF Pride parade (last Sunday of June)
1,2022-06-26,Sunday,135938.0,62336.0,2.18,SF Pride parade (last Sunday of June)
2,2016-02-06,Saturday,413097.0,198662.0,2.08,"Super Bowl 50 week, Super Bowl City in SF"
3,2019-06-30,Sunday,242100.0,117165.0,2.07,SF Pride parade (last Sunday of June)
4,2013-06-30,Sunday,300741.0,147612.0,2.04,SF Pride parade (last Sunday of June)
5,2017-01-21,Saturday,348145.0,177409.0,1.96,Women's March
6,2018-06-24,Sunday,248828.0,127882.0,1.95,SF Pride parade (last Sunday of June)
7,2011-06-26,Sunday,253174.0,132489.0,1.91,SF Pride parade (last Sunday of June)
8,2012-06-24,Sunday,266868.0,145152.0,1.84,SF Pride parade (last Sunday of June)
9,2017-06-25,Sunday,250259.0,137002.0,1.83,SF Pride parade (last Sunday of June)


เฉพาะวันทำงาน

In [24]:
top_surges("is_workday")

,date,day_name,day_trips,usual_trips,ratio,event
0,2022-06-20,Monday,192757.0,124299.0,1.55,Warriors championship parade
1,2021-12-16,Thursday,123725.0,85615.0,1.45,unknown
2,2012-10-31,Wednesday,570234.0,411720.0,1.39,Giants World Series parade
3,2015-06-19,Friday,546247.0,421724.0,1.30,Warriors championship parade
4,2016-02-05,Friday,526415.0,421100.0,1.25,"Super Bowl 50 week, Super Bowl City in SF"
5,2021-12-15,Wednesday,115921.0,93394.0,1.24,unknown
6,2022-12-14,Wednesday,159846.0,131297.0,1.22,unknown
7,2017-06-15,Thursday,521066.0,428400.0,1.22,Warriors championship parade
8,2021-12-09,Thursday,125373.0,104715.0,1.20,unknown
9,2017-12-08,Friday,420519.0,350991.0,1.20,unknown


In [25]:
biggest = fetch_row(con, "SELECT date, ratio FROM surge ORDER BY ratio DESC LIMIT 1")
biggest_day = biggest[0]

compare_query = f"""
    WITH same_weekday AS (
        SELECT c.date
        FROM calendar AS c
        WHERE c.day_status = 'normal'
          AND NOT c.is_holiday
          AND isodow(c.date) = isodow(DATE '{biggest_day}')
          AND c.date BETWEEN DATE '{biggest_day}' - 28 AND DATE '{biggest_day}' + 28
          AND c.date <> DATE '{biggest_day}'
    )
    SELECT
        s.hour,
        avg(s.trips) FILTER (WHERE s.date = DATE '{biggest_day}')         AS event_day,
        avg(s.trips) FILTER (WHERE s.date IN (SELECT date FROM same_weekday)) AS usual_day
    FROM system_hour AS s
    WHERE s.date = DATE '{biggest_day}' OR s.date IN (SELECT date FROM same_weekday)
    GROUP BY s.hour
    ORDER BY s.hour
"""

event_hours = con.sql(compare_query).df()

usual_line = go.Scatter(
    x=event_hours["hour"],
    y=event_hours["usual_day"],
    mode="lines",
    name="usual " + biggest_day.strftime("%A") + " (±4 weeks)",
    line=dict(color=viz.CONTEXT, width=3),
)
event_line = go.Scatter(
    x=event_hours["hour"],
    y=event_hours["event_day"],
    mode="lines",
    name=biggest_day.strftime("%Y-%m-%d"),
    line=dict(color=viz.CONTRAST, width=3),
)

fig = go.Figure([usual_line, event_line])
fig.update_xaxes(title_text="hour", dtick=2)
fig.update_yaxes(title_text="trips", tickformat="~s")

title = f"{event_name(biggest_day)}: {biggest[1]:.1f}x a usual {biggest_day.strftime('%A')}"
subtitle = f"Trips per hour on {biggest_day.strftime('%Y-%m-%d')} vs the average of the same weekday within ±4 weeks"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_biggest_event_day")
fig

### ข้อสังเกต (ส่วนที่ 2)

- ช่วง COVID เป็น structural break ชัดเจน ผู้โดยสารวันทำงานช่วง ม.ค.-เม.ย. 2023 อยู่ที่ 151,896 ทริปต่อวัน เท่ากับ 37% ของปี 2019 (409,262) ส่วนวันหยุดฟื้นได้ดีกว่า อยู่ที่ 51%
- ผู้โดยสารวันหยุดลดลงตั้งแต่ก่อน COVID จาก ~180,000 ทริปต่อวันในปี 2013-2014 เหลือ 137,198 ในปี 2019 (-24%) ขณะที่วันทำงานลดแค่ ~5% จากจุดสูงสุดปี 2016 (432,170)
- วันทำงานยังเป็น double peak เหมือนเดิม แต่สัดส่วนช่วงเช้า (6-9 โมง) ลดจาก 33.7% เหลือ 30.6% ของวัน ส่วนช่วงเย็น (15-18 น.) ยังอยู่ที่ ~35% ช่วงกลางวัน (11-14 น.) มีสัดส่วนสูงขึ้น และชั่วโมง 18:00 ลดจาก 10.9% เหลือ 9.1% คนกลับบ้านเร็วขึ้น
- วันหยุดสุดสัปดาห์รูปทรงแทบไม่เปลี่ยน
- หลัง COVID วันจันทร์และศุกร์เบากว่าอังคาร-พฤหัสชัดเจนกว่าเดิม จันทร์อยู่ที่ 85 และศุกร์ 83 เทียบกับอังคาร-พฤหัส (ปี 2019 เป็น 95 และ 92) ตรงกับรูปแบบ hybrid work ที่เข้าออฟฟิศกลางสัปดาห์
- ฤดูกาลรายเดือนมีน้อย ส่วนใหญ่อยู่ในช่วง ±5% ยกเว้นธันวาคม (92) และมกราคม (96) ช่วงที่สูงที่สุดคือกันยายน-ตุลาคม (104-105)
- วันพีคผิดปกติส่วนใหญ่เป็นเหตุการณ์ที่เกิดซ้ำทุกปี 8 ใน 10 อันดับแรกคือวัน SF Pride (อาทิตย์สุดท้ายของเดือนมิถุนายน) ที่มีผู้โดยสาร ~2 เท่าของวันอาทิตย์ปกติ ที่เหลือคือ Super Bowl 50 และ Women's March
- วันทำงานที่พีคคือขบวนพาเหรดแชมป์ Giants และ Warriors (1.2-1.55 เท่า) ส่วนวันพุธ-พฤหัสในเดือนธันวาคมปี 2021-2022 ยังไม่ทราบสาเหตุ

ใช้ต่อที่ไหน
- Q1: ดูการฟื้นตัวแยกวันในสัปดาห์และแยกวันทำงาน/วันหยุด เพราะต่างกันมาก (37% กับ 51%)
- Q2: peak เช้าเล็กลงและกลางวันหนาแน่นขึ้น เป็นประเด็นหลักเรื่องการปรับ headway
- M1: feature ที่สำคัญคือ ชั่วโมง, วันในสัปดาห์ (จันทร์/ศุกร์ต้องแยกจากกลางสัปดาห์), วันหยุด ส่วนเดือนมีผลน้อย และควรมี flag วัน Pride เพราะเกิดทุกปีในวันที่คาดได้
- M3: วัน Pride และขบวนพาเหรดเป็นตัวอย่าง surge ที่รู้คำตอบอยู่แล้ว ใช้ตรวจว่าระบบตรวจจับเหตุการณ์จับได้ไหม

## 3. สถานี

ใช้เฉพาะวันทำงาน (normal, ไม่ใช่วันหยุด) เทียบ ม.ค.-เม.ย. 2019 กับ ม.ค.-เม.ย. 2023
ผู้โดยสารของสถานี = entries + exits ต่อวัน
MLPT และ BERY เปิดปี 2020 จึงไม่มีค่าปี 2019

### 3.1 อันดับสถานี

In [26]:
station_period_query = f"""
    CREATE OR REPLACE TEMP TABLE station_period AS
    WITH days_in_period AS (
        SELECT
            d.station::VARCHAR        AS station,
            {period_case("d.date")} AS period,
            d.entries,
            d.exits
        FROM station_day AS d
        JOIN calendar AS c ON c.date = d.date
        WHERE c.day_status = 'normal' AND c.is_workday
    )
    SELECT
        station,
        period,
        avg(entries)          AS entries,
        avg(exits)            AS exits,
        avg(entries + exits)  AS riders
    FROM days_in_period
    WHERE period IS NOT NULL
    GROUP BY station, period
"""

with stage("03.station_period"):
    con.execute(station_period_query)

ranking = con.sql("""
    WITH before AS (
        SELECT station, riders FROM station_period WHERE period = 'Jan-Apr 2019'
    ),
    after AS (
        SELECT station, riders FROM station_period WHERE period = 'Jan-Apr 2023'
    )
    SELECT
        s.station,
        s.name,
        s.corridor,
        round(b.riders)                             AS riders_2019,
        round(a.riders)                             AS riders_2023,
        rank() OVER (ORDER BY b.riders DESC)        AS rank_2019,
        rank() OVER (ORDER BY a.riders DESC)        AS rank_2023,
        round(100 * a.riders / b.riders)            AS pct_of_2019
    FROM stations AS s
    LEFT JOIN before AS b ON b.station = s.station
    LEFT JOIN after AS a ON a.station = s.station
    ORDER BY a.riders DESC
""").df()

ranking

[perf] 03.station_period: 0.01s | 3.54 cores | peak RAM 0.484 GB


,station,name,corridor,riders_2019,riders_2023,rank_2019,rank_2023,pct_of_2019
0,EMBR,Embarcadero,SF Downtown,89619.0,28550.0,2,1,32.0
1,MONT,Montgomery St,SF Downtown,89963.0,24143.0,1,2,27.0
2,POWL,Powell St,SF Downtown,54851.0,21917.0,3,3,40.0
3,CIVC,Civic Center/UN Plaza,SF Downtown,47218.0,16948.0,4,4,36.0
4,16TH,16th St Mission,SF Mission/South,25363.0,10327.0,7,5,41.0
5,24TH,24th St Mission,SF Mission/South,24458.0,10112.0,8,6,41.0
6,DBRK,Downtown Berkeley,Berkeley-Richmond line,21897.0,9816.0,9,7,45.0
7,12TH,12th St Oakland City Center,Oakland Core,27919.0,8990.0,5,8,32.0
8,19TH,19th St Oakland,Oakland Core,27486.0,8820.0,6,9,32.0
9,DALY,Daly City,Peninsula,18810.0,8397.0,11,10,45.0


In [27]:
cumulative = con.sql("""
    SELECT
        period,
        row_number() OVER (PARTITION BY period ORDER BY riders DESC) AS station_rank,
        100 * sum(riders) OVER (PARTITION BY period ORDER BY riders DESC)
            / sum(riders) OVER (PARTITION BY period) AS cumulative_pct
    FROM station_period
    ORDER BY period, station_rank
""").df()

top10_share = {}
for _, row in cumulative.iterrows():
    if row["station_rank"] == 10:
        top10_share[row["period"]] = row["cumulative_pct"]

for period in periods:
    print(f"{period}: top 10 stations = {top10_share[period]:.1f}% of riders")

Jan-Apr 2019: top 10 stations = 52.5% of riders
Jan-Apr 2023: top 10 stations = 48.7% of riders


In [28]:
cumulative_colors = {"Jan-Apr 2019": viz.CONTEXT, "Jan-Apr 2023": viz.ACCENT}

fig = go.Figure()
for period, color in cumulative_colors.items():
    rows = cumulative[cumulative["period"] == period]
    fig.add_trace(go.Scatter(
        x=rows["station_rank"],
        y=rows["cumulative_pct"],
        mode="lines+markers",
        name=period,
        line=dict(color=color, width=3),
        marker=dict(size=5),
        hovertemplate="top %{x} stations: %{y:.1f}% of riders<extra>%{fullData.name}</extra>",
    ))

fig.add_vline(x=10, line=dict(color=viz.MUTED, width=1, dash="dot"))
fig.update_layout(hovermode="closest")
fig.update_xaxes(title_text="number of stations (largest first)")
fig.update_yaxes(title_text="cumulative % of riders", ticksuffix="%", range=[0, 101])

before = top10_share["Jan-Apr 2019"]
after = top10_share["Jan-Apr 2023"]
title = f"The top 10 stations carried {before:.0f}% of riders in 2019 and {after:.0f}% in 2023"
subtitle = "Cumulative share of workday riders (entries + exits), stations sorted from largest, normal workdays Jan-Apr"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_station_concentration")
fig

In [29]:
top15 = ranking.sort_values("riders_2019", ascending=False).head(15)
top15 = top15.sort_values("riders_2019", ascending=True)

labels = []
for _, row in top15.iterrows():
    labels.append(f"{row['station']}  {row['name']}")

bars_2019 = go.Bar(
    name="Jan-Apr 2019",
    y=labels,
    x=top15["riders_2019"],
    orientation="h",
    marker_color=viz.CONTEXT,
)
bars_2023 = go.Bar(
    name="Jan-Apr 2023",
    y=labels,
    x=top15["riders_2023"],
    orientation="h",
    marker_color=viz.ACCENT,
    customdata=top15["pct_of_2019"],
    hovertemplate="%{y}: %{x:,.0f} riders per workday (%{customdata:.0f}% of 2019)<extra></extra>",
)

fig = go.Figure([bars_2019, bars_2023])
fig.update_traces(selector=dict(name="Jan-Apr 2019"), hovertemplate="%{y}: %{x:,.0f} riders per workday<extra></extra>")
fig.update_layout(barmode="group", hovermode="closest", margin=dict(l=260))
fig.update_xaxes(title_text="riders per workday (entries + exits)", tickformat="~s", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False)

losses = top15.copy()
losses["lost"] = losses["riders_2019"] - losses["riders_2023"]
losses = losses.sort_values("lost", ascending=False)
first = losses.iloc[0]
second = losses.iloc[1]

title = f"Downtown SF lost the most riders: {first['station']} -{first['lost'] / 1000:.0f}k and {second['station']} -{second['lost'] / 1000:.0f}k per workday"
subtitle = "15 busiest stations of 2019, average riders per normal workday, Jan-Apr 2019 vs Jan-Apr 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_top_stations", height=680)
fig

### 3.2 ขาเข้า vs ขาออกช่วงเช้า

exit share = exits / (entries + exits) ช่วง 6:00-9:59 ของวันทำงาน
- ค่าสูง (มากกว่า 50%) คนออกจากสถานีมากกว่าเข้า แปลว่าเป็นปลายทางตอนเช้า มักเป็นย่านที่ทำงาน
- ค่าต่ำ คนเข้าสถานีมากกว่า แปลว่าเป็นต้นทางตอนเช้า มักเป็นย่านที่พัก

In [30]:
am_query = f"""
    CREATE OR REPLACE TEMP TABLE am_balance AS
    WITH morning AS (
        SELECT
            h.station::VARCHAR        AS station,
            {period_case("h.date")} AS period,
            h.entries,
            h.exits
        FROM station_hour AS h
        JOIN calendar AS c ON c.date = h.date
        WHERE c.day_status = 'normal' AND c.is_workday AND h.hour BETWEEN 6 AND 9
    )
    SELECT
        station,
        period,
        sum(exits) / (sum(entries) + sum(exits)) AS exit_share
    FROM morning
    WHERE period IS NOT NULL
    GROUP BY station, period
"""

con.execute(am_query)

am_balance = con.sql("""
    SELECT
        s.station,
        s.name,
        s.corridor,
        round(100 * b.exit_share, 1) AS exit_share_2019,
        round(100 * a.exit_share, 1) AS exit_share_2023
    FROM stations AS s
    LEFT JOIN am_balance AS b ON b.station = s.station AND b.period = 'Jan-Apr 2019'
    LEFT JOIN am_balance AS a ON a.station = s.station AND a.period = 'Jan-Apr 2023'
    ORDER BY coalesce(b.exit_share, a.exit_share)
""").df()

destinations = am_balance[am_balance["exit_share_2019"] > 50]
print("morning destinations in 2019 (exit share > 50%):", ", ".join(destinations["station"]))
am_balance

morning destinations in 2019 (exit share > 50%): OAKL, 16TH, 19TH, DBRK, 12TH, SFIA, CIVC, POWL, EMBR, MONT


,station,name,corridor,exit_share_2019,exit_share_2023
0,PITT,Pittsburg/Bay Point,Concord-Antioch line,4.7,7.3
1,ANTC,Antioch,Concord-Antioch line,5.1,6.5
2,NCON,North Concord/Martinez,Concord-Antioch line,6.3,15.1
3,CAST,Castro Valley,Dublin line,7.1,12.4
4,COLM,Colma,Peninsula,8.0,18.0
5,PCTR,Pittsburg Center,Concord-Antioch line,8.5,14.2
6,PLZA,El Cerrito Plaza,Berkeley-Richmond line,8.7,15.2
7,PHIL,Pleasant Hill/Contra Costa Centre,Concord-Antioch line,8.8,13.9
8,DELN,El Cerrito del Norte,Berkeley-Richmond line,9.0,13.0
9,ORIN,Orinda,Concord-Antioch line,9.8,15.4


In [31]:
segment_x = []
segment_y = []
for _, row in am_balance.iterrows():
    segment_x.extend([row["exit_share_2019"], row["exit_share_2023"], None])
    segment_y.extend([row["station"], row["station"], None])

segments = go.Scatter(
    x=segment_x,
    y=segment_y,
    mode="lines",
    line=dict(color=viz.GRID, width=3),
    showlegend=False,
    hoverinfo="skip",
)
points_2019 = go.Scatter(
    x=am_balance["exit_share_2019"],
    y=am_balance["station"],
    mode="markers",
    name="Jan-Apr 2019",
    marker=dict(color=viz.CONTEXT, size=9),
    customdata=am_balance["name"],
    hovertemplate="%{y} %{customdata}: %{x:.1f}% exits<extra>2019</extra>",
)
points_2023 = go.Scatter(
    x=am_balance["exit_share_2023"],
    y=am_balance["station"],
    mode="markers",
    name="Jan-Apr 2023",
    marker=dict(color=viz.ACCENT, size=9),
    customdata=am_balance["name"],
    hovertemplate="%{y} %{customdata}: %{x:.1f}% exits<extra>2023</extra>",
)

fig = go.Figure([segments, points_2019, points_2023])
fig.add_vline(x=50, line=dict(color=viz.MUTED, width=1, dash="dot"))
fig.update_layout(hovermode="closest", margin=dict(t=150))
fig.update_xaxes(title_text="exits as % of entries + exits, 6:00-9:59", ticksuffix="%", showgrid=True, gridcolor=viz.GRID)
fig.update_yaxes(showgrid=False, tickfont=dict(size=11))

title = "In the morning only about 10 stations take in more riders than they send out"
subtitle = "Morning exit share per station, normal workdays. Right of 50% = morning destination (work area)"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_am_exit_share", height=1000)
fig

### 3.3 profile รายชั่วโมงของสถานีตัวอย่าง

สัดส่วนของ entries และ exits ในแต่ละชั่วโมง (ของทั้งวัน) วันทำงาน
- MONT downtown SF
- DUBL ปลายสายฝั่งตะวันออก ย่านที่พัก
- SFIA สนามบิน
- 16TH ย่าน Mission ทั้งที่พักและที่เที่ยว

In [32]:
sample_stations = ["MONT", "DUBL", "SFIA", "16TH"]

station_list = []
for station in sample_stations:
    station_list.append(f"'{station}'")
station_sql = ", ".join(station_list)

profile_query = f"""
    WITH hours_in_period AS (
        SELECT
            h.station::VARCHAR        AS station,
            {period_case("h.date")} AS period,
            h.hour,
            h.entries,
            h.exits
        FROM station_hour AS h
        JOIN calendar AS c ON c.date = h.date
        WHERE c.day_status = 'normal'
          AND c.is_workday
          AND h.station::VARCHAR IN ({station_sql})
    ),
    hourly AS (
        SELECT station, period, hour, avg(entries) AS entries, avg(exits) AS exits
        FROM hours_in_period
        WHERE period IS NOT NULL
        GROUP BY station, period, hour
    )
    SELECT
        station,
        period,
        hour,
        100 * entries / sum(entries) OVER (PARTITION BY station, period) AS entries_pct,
        100 * exits / sum(exits) OVER (PARTITION BY station, period)     AS exits_pct
    FROM hourly
    ORDER BY station, period, hour
"""

profiles = con.sql(profile_query).df()

In [33]:
station_names = {}
for _, row in ranking.iterrows():
    station_names[row["station"]] = row["name"]

subplot_titles = []
for station in sample_stations:
    subplot_titles.append(f"{station}  {station_names[station]}")

fig = make_subplots(rows=2, cols=2, subplot_titles=subplot_titles, shared_xaxes=True, vertical_spacing=0.12)

line_specs = [
    ("entries_pct", "Jan-Apr 2019", "Entries 2019", dict(color=viz.ACCENT, width=2, dash="dot")),
    ("exits_pct", "Jan-Apr 2019", "Exits 2019", dict(color=viz.CONTRAST, width=2, dash="dot")),
    ("entries_pct", "Jan-Apr 2023", "Entries 2023", dict(color=viz.ACCENT, width=3)),
    ("exits_pct", "Jan-Apr 2023", "Exits 2023", dict(color=viz.CONTRAST, width=3)),
]

for position, station in enumerate(sample_stations):
    row = position // 2 + 1
    col = position % 2 + 1
    for column, period, label, style in line_specs:
        rows = profiles[(profiles["station"] == station) & (profiles["period"] == period)]
        fig.add_trace(
            go.Scatter(
                x=rows["hour"],
                y=rows[column],
                mode="lines",
                name=label,
                line=style,
                legendgroup=label,
                showlegend=(position == 0),
                hovertemplate="%{x}:00 - %{y:.1f}% of the day<extra>" + station + " " + label + "</extra>",
            ),
            row=row,
            col=col,
        )

fig.update_xaxes(dtick=3)
fig.update_yaxes(ticksuffix="%")
fig.update_layout(margin=dict(t=150))

title = "Each type of station has its own daily signature"
subtitle = "Share of the day's entries (blue) and exits (orange) per hour, normal workdays. Dotted = Jan-Apr 2019, solid = Jan-Apr 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_station_profiles", height=700)
fig

### ข้อสังเกต (ส่วนที่ 3)

- ผู้โดยสารกระจุกที่สถานีใหญ่น้อยลง top 10 สถานีเคยรวมกันได้ 52.5% ของผู้โดยสารวันทำงานในปี 2019 เหลือ 48.7% ในปี 2023 เพราะสถานีใหญ่หายไปมากกว่าสถานีอื่น
- 4 สถานี downtown SF (MONT, EMBR, POWL, CIVC) เสียผู้โดยสารมากที่สุดในเชิงจำนวน เหลือ 27-40% ของปี 2019 MONT หายไป ~66,000 และ EMBR ~61,000 คนต่อวันทำงาน corridor SF Downtown รวมกันหายไป ~190,000 คนต่อวัน คิดเป็น ~37% ของที่หายไปทั้งระบบ
- EMBR แซง MONT ขึ้นเป็นอันดับ 1 ส่วน downtown Oakland (12TH, 19TH) เหลือแค่ 32% ตกจากอันดับ 5-6 ไปอันดับ 8-9
- ถ้าดูเป็น % corridor ที่ฟื้นช้าที่สุดคือ Dublin line (32%) และ SF Downtown (33%) ส่วน Berkeley-Richmond line ฟื้นดีที่สุด (43%)
- ช่วงเช้ามีแค่ 10 สถานีที่คนออกมากกว่าเข้า (ปลายทางตอนเช้า) คือ MONT, EMBR, POWL, CIVC, SFIA, 12TH, DBRK, 19TH, 16TH, OAKL สถานีที่เหลือเป็นต้นทาง ทริปตอนเช้าจึงวิ่งเข้าหาสถานีกลุ่มเล็กๆ นี้
- ปี 2023 สถานีต้นทางเกือบทุกสถานีมี exit share สูงขึ้น แปลว่าคนที่ออกจากบ้านไปทำงานตอนเช้าลดลงมากกว่าการเดินทางแบบอื่น ส่วน downtown Oakland (19TH 63% เหลือ 53%, 12TH 74% เหลือ 64%) เป็นปลายทางน้อยลง
- WARM เปลี่ยนมากที่สุด จาก 26% เป็น 52% กลายเป็นปลายทางตอนเช้า ยังไม่รู้สาเหตุ ต้องดูต่อใน Q4 (ช่วงเดียวกับที่ MLPT/BERY เปิด)
- profile รายชั่วโมงต่างกันชัดตามประเภทสถานี MONT คนออกพีคตอน 8 โมง (~25% ของคนออกทั้งวันในชั่วโมงเดียว) และคนเข้าพีคตอน 5 โมงเย็น DUBL กลับด้านกัน SFIA กระจายทั้งวันไม่มีพีคคมๆ ส่วน 16TH มีพีคทั้งเข้าและออกทั้งสองช่วง
- รูปทรงปี 2023 ใกล้เคียงปี 2019 แต่พีคต่ำลงเล็กน้อย และ MONT คนเข้าสถานีช่วงเย็นเลื่อนเร็วขึ้น (16:00 สูงขึ้น 18:00 ต่ำลง)

ใช้ต่อที่ไหน
- M2: profile รายชั่วโมงของ entries/exits และ exit share ช่วงเช้าแยกประเภทสถานีได้ชัด เป็น feature ที่ใช้ได้ตรงๆ
- Q1: วิเคราะห์การฟื้นตัวราย corridor และรายสถานี โดยเฉพาะ downtown SF กับ Dublin line
- Q3: flow ตอนเช้าวิ่งเข้าหา 10 สถานี เป็นจุดเริ่มดูโหลดตามทิศทาง
- Q4: ดูเรื่อง WARM ต่อ

## 4. Origin-Destination

ใช้ od_month (วัน normal เติมศูนย์แล้ว) หารด้วยจำนวนวันจาก month_days ได้ทริปเฉลี่ยต่อวันทำงานของแต่ละคู่ OD
เทียบ ม.ค.-เม.ย. 2019 กับ ม.ค.-เม.ย. 2023

### 4.1 OD matrix

In [34]:
od_period_query = f"""
    CREATE OR REPLACE TEMP TABLE od_period AS
    WITH workday_months AS (
        SELECT
            {period_case("o.month")} AS period,
            o.origin::VARCHAR          AS origin,
            o.dest::VARCHAR            AS dest,
            o.trips,
            d.days
        FROM od_month AS o
        JOIN month_days AS d ON d.month = o.month AND d.day_type = o.day_type
        WHERE o.day_type = 'workday'
    )
    SELECT
        period,
        origin,
        dest,
        sum(trips) / sum(days) AS trips_per_day
    FROM workday_months
    WHERE period IS NOT NULL
    GROUP BY period, origin, dest
"""

with stage("03.od_period"):
    con.execute(od_period_query)

con.sql("""
    SELECT period, count(*) AS pairs, round(sum(trips_per_day)) AS trips_per_day
    FROM od_period
    GROUP BY period
    ORDER BY period
""").df()

[perf] 03.od_period: 0.01s | 2.7 cores | peak RAM 0.492 GB


,period,pairs,trips_per_day
0,Jan-Apr 2019,2304,409855.0
1,Jan-Apr 2023,2500,151896.0


เรียงสถานีตาม corridor ใน `stations.py` (ฝั่งตะวันตกของอ่าวก่อน แล้วตามด้วยฝั่งตะวันออก)
สีเป็น log scale เพราะคู่ใหญ่สุดกับเล็กสุดต่างกันหลายพันเท่า
เส้นประแบ่งฝั่งอ่าว บล็อกมุมขวาบนและซ้ายล่างคือทริปที่ข้าม Transbay Tube

In [35]:
station_order = list(STATIONS)

west_count = 0
for code, (name, corridor, side) in STATIONS.items():
    if side == "west":
        west_count += 1

od_rows = con.sql("""
    SELECT
        period,
        origin,
        dest,
        trips_per_day,
        CASE WHEN trips_per_day > 0 THEN log10(trips_per_day) END AS log_trips
    FROM od_period
""").df()


def od_matrix(period, column):
    values = {}
    for _, row in od_rows[od_rows["period"] == period].iterrows():
        values[(row["origin"], row["dest"])] = row[column]

    matrix = []
    for origin in station_order:
        matrix_row = []
        for dest in station_order:
            matrix_row.append(values.get((origin, dest)))
        matrix.append(matrix_row)
    return matrix


print("stations on the west side:", west_count)

stations on the west side: 14


In [36]:
fig = make_subplots(rows=1, cols=2, subplot_titles=list(periods), horizontal_spacing=0.06)

column = 1
for period in periods:
    fig.add_trace(
        go.Heatmap(
            z=od_matrix(period, "log_trips"),
            x=station_order,
            y=station_order,
            customdata=od_matrix(period, "trips_per_day"),
            colorscale=viz.SEQUENTIAL,
            zmin=-1,
            zmax=4,
            showscale=(column == 2),
            colorbar=dict(
                title="trips per day",
                tickvals=[-1, 0, 1, 2, 3, 4],
                ticktext=["0.1", "1", "10", "100", "1k", "10k"],
            ),
            hovertemplate="%{y} to %{x}: %{customdata:,.1f} trips per workday<extra></extra>",
        ),
        row=1,
        col=column,
    )
    column += 1

divider = west_count - 0.5
for axis in ["x", "x2"]:
    y_axis = axis.replace("x", "y")
    fig.add_shape(type="line", xref=axis, yref=y_axis, x0=divider, x1=divider, y0=-0.5, y1=len(station_order) - 0.5,
                  line=dict(color=viz.CONTRAST, width=1, dash="dot"))
    fig.add_shape(type="line", xref=axis, yref=y_axis, x0=-0.5, x1=len(station_order) - 0.5, y0=divider, y1=divider,
                  line=dict(color=viz.CONTRAST, width=1, dash="dot"))

fig.update_xaxes(tickfont=dict(size=8), tickangle=-90, title_text="destination")
fig.update_yaxes(tickfont=dict(size=8), autorange="reversed", title_text="origin")
fig.update_layout(hovermode="closest", margin=dict(t=130))

title = "The OD pattern kept its basic shape while the level dropped"
subtitle = "Average trips per workday for every origin-destination pair, log colour scale. Dotted lines split west and east of the Bay"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_od_matrix", width=1200, height=700)
fig

### 4.2 ความเข้มข้นของ flow

เรียงคู่ OD จากใหญ่ไปเล็ก แล้วดูว่าต้องใช้กี่คู่ถึงจะรวมได้ 50% และ 80% ของทริป

In [37]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE od_cumulative AS
    SELECT
        period,
        origin,
        dest,
        trips_per_day,
        row_number() OVER (PARTITION BY period ORDER BY trips_per_day DESC) AS pair_rank,
        count(*) OVER (PARTITION BY period)                                AS total_pairs,
        100 * sum(trips_per_day) OVER (PARTITION BY period ORDER BY trips_per_day DESC
                                       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
            / sum(trips_per_day) OVER (PARTITION BY period)                AS cumulative_pct
    FROM od_period
""")

concentration = con.sql("""
    SELECT
        period,
        max(total_pairs)                                        AS total_pairs,
        min(pair_rank) FILTER (WHERE cumulative_pct >= 50)      AS pairs_for_50pct,
        min(pair_rank) FILTER (WHERE cumulative_pct >= 80)      AS pairs_for_80pct,
        round(100 * min(pair_rank) FILTER (WHERE cumulative_pct >= 50) / max(total_pairs), 1) AS pct_pairs_for_50pct,
        round(100 * min(pair_rank) FILTER (WHERE cumulative_pct >= 80) / max(total_pairs), 1) AS pct_pairs_for_80pct
    FROM od_cumulative
    GROUP BY period
    ORDER BY period
""").df()

concentration

,period,total_pairs,pairs_for_50pct,pairs_for_80pct,pct_pairs_for_50pct,pct_pairs_for_80pct
0,Jan-Apr 2019,2304,184,541,8.0,23.5
1,Jan-Apr 2023,2500,233,698,9.3,27.9


In [38]:
curve = con.sql("""
    SELECT period, 100 * pair_rank / total_pairs AS pct_pairs, cumulative_pct
    FROM od_cumulative
    ORDER BY period, pair_rank
""").df()

curve_colors = {"Jan-Apr 2019": viz.CONTEXT, "Jan-Apr 2023": viz.ACCENT}

fig = go.Figure()
for period, color in curve_colors.items():
    rows = curve[curve["period"] == period]
    fig.add_trace(go.Scatter(
        x=rows["pct_pairs"],
        y=rows["cumulative_pct"],
        mode="lines",
        name=period,
        line=dict(color=color, width=3),
        hovertemplate="top %{x:.1f}% of pairs: %{y:.1f}% of trips<extra>%{fullData.name}</extra>",
    ))

fig.add_hline(y=50, line=dict(color=viz.MUTED, width=1, dash="dot"))
fig.add_hline(y=80, line=dict(color=viz.MUTED, width=1, dash="dot"))
fig.update_layout(hovermode="closest")
fig.update_xaxes(title_text="% of OD pairs (largest first)", ticksuffix="%")
fig.update_yaxes(title_text="cumulative % of trips", ticksuffix="%", range=[0, 101])

share_2019 = concentration.set_index("period").loc["Jan-Apr 2019"]
title = f"{share_2019['pct_pairs_for_50pct']:.0f}% of OD pairs carry half of all trips"
subtitle = "Cumulative share of workday trips by OD pair, pairs sorted from largest, Jan-Apr 2019 vs Jan-Apr 2023"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_od_concentration")
fig

10 คู่ OD ที่ใหญ่ที่สุดของแต่ละช่วง

In [39]:
top_pairs = con.sql("""
    SELECT
        period,
        pair_rank,
        origin || ' to ' || dest  AS pair,
        round(trips_per_day)      AS trips_per_day
    FROM od_cumulative
    WHERE pair_rank <= 10
    ORDER BY period, pair_rank
""").df()

top_pairs.pivot(index="pair_rank", columns="period", values=["pair", "trips_per_day"])

pair               trips_per_day             
period     Jan-Apr 2019  Jan-Apr 2023  Jan-Apr 2019 Jan-Apr 2023
pair_rank                                                       
1          BALB to MONT  POWL to BALB        2653.0        846.0
2          24TH to MONT  WOAK to EMBR        2472.0        792.0
3          DUBL to EMBR  BALB to POWL        2459.0        735.0
4          WOAK to EMBR  POWL to 24TH        2253.0        717.0
5          POWL to BALB  24TH to MONT        2241.0        711.0
6          MONT to 24TH  BALB to MONT        2107.0        691.0
7          PHIL to EMBR  EMBR to WOAK        2092.0        679.0
8          DALY to MONT  24TH to EMBR        2071.0        670.0
9          EMBR to DUBL  DALY to MONT        2038.0        644.0
10         24TH to EMBR  DUBL to EMBR        1946.0        642.0

### 4.3 ทริปข้าม Transbay Tube

ทริปข้าม Tube = ต้นทางกับปลายทางอยู่คนละฝั่งอ่าว (ใช้คอลัมน์ side ใน stations)
ดูสัดส่วนต่อทริปทั้งหมดรายเดือน แยกวันทำงานกับวันหยุด ไม่นับ พ.ค. 2023

In [40]:
transbay_query = """
    SELECT
        o.month,
        o.day_type,
        sum(o.trips) FILTER (WHERE so.side <> sd.side) AS crossing_trips,
        sum(o.trips)                                   AS all_trips,
        100 * sum(o.trips) FILTER (WHERE so.side <> sd.side) / sum(o.trips) AS crossing_pct
    FROM od_month AS o
    JOIN stations AS so ON so.station = o.origin::VARCHAR
    JOIN stations AS sd ON sd.station = o.dest::VARCHAR
    WHERE o.month < DATE '2023-05-01'
    GROUP BY o.month, o.day_type
"""

with stage("03.transbay"):
    con.execute(f"CREATE OR REPLACE TEMP TABLE transbay_months AS {transbay_query}")
    transbay = con.sql("SELECT * FROM transbay_months ORDER BY month, day_type").df()

fig = go.Figure()
for day_type, style in line_styles.items():
    rows = transbay[transbay["day_type"] == day_type]
    fig.add_trace(go.Scatter(
        x=rows["month"],
        y=rows["crossing_pct"],
        mode="lines",
        name=line_names[day_type],
        line=style,
        hovertemplate="%{x|%b %Y}: %{y:.1f}% of trips cross the Bay<extra>%{fullData.name}</extra>",
    ))

viz.vline(fig, "2020-03-09", "COVID break", y=1.02)
fig.update_yaxes(title_text="% of trips crossing the Bay", ticksuffix="%", rangemode="tozero")

crossing_2019 = fetch_value(con, """
    SELECT 100 * sum(crossing_trips) / sum(all_trips) FROM transbay_months
    WHERE day_type = 'workday' AND month BETWEEN DATE '2019-01-01' AND DATE '2019-04-01'
""")
crossing_2023 = fetch_value(con, """
    SELECT 100 * sum(crossing_trips) / sum(all_trips) FROM transbay_months
    WHERE day_type = 'workday' AND month BETWEEN DATE '2023-01-01' AND DATE '2023-04-01'
""")

title = f"Workday trips crossing the Bay: {crossing_2019:.0f}% in early 2019, {crossing_2023:.0f}% in early 2023"
subtitle = "Trips whose origin and destination are on different sides of the Bay, % of all trips per month, normal days"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_transbay_share")
fig

[perf] 03.transbay: 0.01s | 3.29 cores | peak RAM 0.503 GB


แยกตามทิศทาง ทริปเฉลี่ยต่อวันทำงาน

In [41]:
direction_query = f"""
    WITH workday_months AS (
        SELECT
            {period_case("o.month")} AS period,
            so.side AS origin_side,
            sd.side AS dest_side,
            o.trips,
            o.month
        FROM od_month AS o
        JOIN stations AS so ON so.station = o.origin::VARCHAR
        JOIN stations AS sd ON sd.station = o.dest::VARCHAR
        WHERE o.day_type = 'workday'
    ),
    trips_by_direction AS (
        SELECT period, origin_side, dest_side, sum(trips) AS trips
        FROM workday_months
        WHERE period IS NOT NULL
        GROUP BY period, origin_side, dest_side
    ),
    days_in_period AS (
        SELECT {period_case("month")} AS period, sum(days) AS days
        FROM month_days
        WHERE day_type = 'workday'
        GROUP BY period
    )
    SELECT
        t.period,
        t.origin_side || ' to ' || t.dest_side       AS direction,
        round(t.trips / d.days)                      AS trips_per_day,
        round(100 * t.trips / sum(t.trips) OVER (PARTITION BY t.period), 1) AS pct_of_trips
    FROM trips_by_direction AS t
    JOIN days_in_period AS d ON d.period = t.period
    ORDER BY t.period, direction
"""

directions = con.sql(direction_query).df()
directions.pivot(index="direction", columns="period", values=["trips_per_day", "pct_of_trips"])

trips_per_day              pct_of_trips             
period        Jan-Apr 2019 Jan-Apr 2023 Jan-Apr 2019 Jan-Apr 2023
direction                                                        
east to east       79512.0      36502.0         19.4         24.0
east to west      113886.0      39657.0         27.8         26.1
west to east      113978.0      38946.0         27.8         25.6
west to west      102479.0      36792.0         25.0         24.2

### ข้อสังเกต (ส่วนที่ 4)

- OD matrix ปี 2023 ยังมีโครงเดิม คือแถวและคอลัมน์ของ downtown SF (EMBR, MONT, POWL, CIVC) เข้มที่สุด และเห็นบล็อกของแต่ละ corridor ตามแนวทแยง แต่ทริประหว่างสถานีชานเมืองที่อยู่ไกลกัน (เช่น สาย Concord-Antioch ไป Peninsula) จางลงชัดเจน
- flow กระจุกตัวมาก ปี 2019 ใช้แค่ 184 คู่ (8.0% ของคู่ทั้งหมด) ก็รวมได้ครึ่งหนึ่งของทริป และ 541 คู่ (23.5%) ได้ 80% ปี 2023 กระจายขึ้นเล็กน้อยเป็น 9.3% และ 27.9%
- top 10 คู่ OD ปี 2019 เกือบทั้งหมดเป็นการเดินทางไป/กลับ downtown SF (BALB, 24TH, DUBL, WOAK, PHIL, DALY ไป MONT/EMBR และขากลับ) คู่ใหญ่สุดมี ~2,650 ทริปต่อวันทำงาน ปี 2023 คู่ใหญ่สุดเหลือ ~850 และเป็นทริปสั้นภายใน SF มากขึ้น (POWL-BALB, POWL-24TH)
- ทริปข้าม Transbay Tube เป็นสัดส่วนสูงขึ้นเรื่อยๆ ก่อน COVID ประมาณ 49% ในปี 2011 เป็น ~55-56% ในปี 2019 (วันทำงาน) หลัง COVID ลดลงเหลือ ~52%
- จำนวนทริปข้าม Tube ต่อวันทำงานลดจาก ~228,000 เหลือ ~79,000 (34% ของปี 2019) แต่ละทิศใกล้เคียงกันทั้งสองช่วง (ตะวันออกไปตะวันตก 113,886 กับ 39,657 / ตะวันตกไปตะวันออก 113,978 กับ 38,946)
- ทริปภายในฝั่งตะวันออกเป็นสัดส่วนเพิ่มขึ้นจาก 19.4% เป็น 24.0% สอดคล้องกับที่ downtown SF ฟื้นช้าที่สุด
- วันหยุดมีช่วงที่สัดส่วนข้ามอ่าวลดลงชั่วคราว (เช่น ปี 2015 และ 2018) ยังไม่ทราบสาเหตุ

ใช้ต่อที่ไหน
- Q3: ทริปข้าม Tube ยังเป็นครึ่งหนึ่งของระบบ ต้องดูต่อว่าทิศไหนและชั่วโมงไหนแน่นที่สุด (ยอดรวมทั้งวันสองทิศเท่ากัน แต่เช้ากับเย็นจะกลับทิศกัน)
- Q1: การเดินทางที่หายไปมากที่สุดคือการเดินทางไป downtown SF จากชานเมือง
- M2: สถานีที่เชื่อมกับ downtown SF มากเป็นสัญญาณของสถานีย่านที่พัก

## 5. ความสัมพันธ์ตามเวลา (สำหรับ M1)

M1 พยากรณ์ entries รายสถานี-ชั่วโมงล่วงหน้า 7 วัน จึงใช้ได้แค่ข้อมูลที่เก่ากว่า 168 ชั่วโมง
ส่วนนี้ดูว่าค่าในอดีตช่วงไหนบอกอนาคตได้ดี และ baseline ง่ายๆ ทำได้แค่ไหน

ใช้เฉพาะวัน normal และไม่แตะช่วง test ของ M1 (ปี 2023)

### 5.1 Autocorrelation ของ system_hour ปี 2019

ชั่วโมงในวันที่ไม่ใช่ normal ใส่เป็นค่าว่าง ไม่เติม 0

In [42]:
system_2019 = con.sql("""
    SELECT
        s.date,
        s.hour,
        CASE WHEN c.day_status = 'normal' THEN s.trips END AS trips
    FROM system_hour AS s
    JOIN calendar AS c ON c.date = s.date
    WHERE c.year = 2019
    ORDER BY s.date, s.hour
""").df()

trips_series = system_2019["trips"].astype(float)

acf_rows = []
for lag in range(1, 337):
    acf_rows.append({"lag_hours": lag, "acf": trips_series.autocorr(lag)})
acf = pd.DataFrame(acf_rows)

acf_by_lag = acf.set_index("lag_hours")["acf"]
for lag in [1, 24, 48, 144, 168, 336]:
    print(f"lag {lag:>3} hours: {acf_by_lag[lag]:.3f}")

lag   1 hours: 0.820
lag  24 hours: 0.787
lag  48 hours: 0.545
lag 144 hours: 0.767
lag 168 hours: 0.946
lag 336 hours: 0.943


In [43]:
acf_line = go.Scatter(
    x=acf["lag_hours"],
    y=acf["acf"],
    mode="lines",
    name="autocorrelation",
    line=dict(color=viz.ACCENT, width=2),
    hovertemplate="lag %{x} hours: %{y:.3f}<extra></extra>",
)

fig = go.Figure(acf_line)
for lag, label in [(24, "1 day"), (168, "1 week"), (336, "2 weeks")]:
    viz.end_label(fig, lag, acf_by_lag[lag], f"{label}: {acf_by_lag[lag]:.2f}", ax=0, ay=-35)

fig.add_hline(y=0, line=dict(color=viz.MUTED, width=1))
fig.update_layout(hovermode="closest", showlegend=False)
fig.update_xaxes(title_text="lag (hours)", dtick=24)
fig.update_yaxes(title_text="autocorrelation", range=[-0.6, 1.05])

title = f"Same hour last week (r = {acf_by_lag[168]:.2f}) beats same hour yesterday (r = {acf_by_lag[24]:.2f})"
subtitle = "Autocorrelation of system trips per hour in 2019, normal days only, lags up to two weeks"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_acf")
fig

### 5.2 entries ของสถานี-ชั่วโมง เทียบกับสัปดาห์ก่อนๆ

correlation ระหว่าง entries กับค่าของสถานีเดิม ชั่วโมงเดิม เมื่อ k สัปดาห์ก่อน (ทั้งสองวันต้องเป็นวัน normal)
เทียบปี 2019 กับช่วง valid ของ M1 (ก.ค.-ธ.ค. 2022)

ข้อควรระวัง: correlation ระดับนี้สูงอยู่แล้วส่วนหนึ่ง เพราะสถานีใหญ่กับเล็กต่างกันมาก ตัวชี้วัดหลักจึงเป็น WAPE ในหัวข้อ 5.3

In [44]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE normal_hours AS
    SELECT h.station, h.date, h.hour, h.entries::DOUBLE AS entries, c.is_workday
    FROM station_hour AS h
    JOIN calendar AS c ON c.date = h.date
    WHERE c.day_status = 'normal'
""")

lag_periods = {
    "2019": ("2019-01-01", "2019-12-31"),
    "Jul-Dec 2022 (M1 valid)": ("2022-07-01", "2022-12-31"),
}

lag_rows = []
with stage("03.lag_correlation"):
    for label, (start, end) in lag_periods.items():
        for weeks in [1, 2, 3, 4, 52]:
            correlation, pairs = fetch_row(con, f"""
                SELECT corr(t.entries, l.entries), count(*)
                FROM normal_hours AS t
                JOIN normal_hours AS l
                    ON l.station = t.station
                   AND l.hour = t.hour
                   AND l.date = t.date - {7 * weeks}
                WHERE t.date BETWEEN DATE '{start}' AND DATE '{end}'
            """)
            lag_rows.append({"period": label, "weeks_back": weeks, "correlation": correlation, "pairs": pairs})

lag_correlation = pd.DataFrame(lag_rows)
lag_correlation.pivot(index="weeks_back", columns="period", values="correlation").round(4)

[perf] 03.lag_correlation: 0.17s | 2.87 cores | peak RAM 0.677 GB


period,2019,Jul-Dec 2022 (M1 valid)
weeks_back,,
1,0.9492,0.9453
2,0.9400,0.9337
3,0.9434,0.9332
4,0.9529,0.9386
52,0.9692,0.9431


In [45]:
lag_colors = {"2019": viz.CONTEXT, "Jul-Dec 2022 (M1 valid)": viz.ACCENT}

fig = go.Figure()
for period, color in lag_colors.items():
    rows = lag_correlation[lag_correlation["period"] == period]
    labels = []
    for weeks in rows["weeks_back"]:
        if weeks == 1:
            labels.append("1 week")
        else:
            labels.append(f"{weeks} weeks")
    fig.add_trace(go.Scatter(
        name=period,
        x=labels,
        y=rows["correlation"],
        mode="lines+markers",
        line=dict(color=color, width=2),
        marker=dict(color=color, size=12),
        hovertemplate="%{x} back: r = %{y:.4f}<extra>%{fullData.name}</extra>",
    ))

fig.update_layout(hovermode="closest")
fig.update_xaxes(title_text="same station and hour, k weeks back")
fig.update_yaxes(title_text="correlation", range=[0.9, 1.0])

title = "Correlation with past weeks stays high; the one-year lag fell the most after COVID"
subtitle = "Correlation of station-hour entries with the same station and hour k weeks earlier, normal days (y axis zoomed to 0.9-1.0)"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_lag_correlation")
fig

### 5.3 Baseline ในช่วง valid ของ M1

ลองพยากรณ์ entries ของ ก.ค.-ธ.ค. 2022 ด้วยวิธีง่ายๆ 3 แบบ
- last week: ค่าของสถานีเดิม ชั่วโมงเดิม เมื่อ 7 วันก่อน
- mean of 4 weeks: ค่าเฉลี่ยของ 7, 14, 21, 28 วันก่อน (เฉพาะวันที่เป็น normal)
- last year: ค่าเมื่อ 364 วันก่อน (วันเดียวกันในสัปดาห์)

WAPE = ผลรวมของ |ค่าจริง - ค่าพยากรณ์| / ผลรวมของค่าจริง
เทียบทั้ง 3 แบบบนแถวชุดเดียวกัน คือแถวที่ทั้ง 3 แบบมีค่า (วันที่ใช้เป็น lag ต้องเป็นวัน normal)

In [46]:
baseline_query = """
    CREATE OR REPLACE TEMP TABLE baseline_eval AS
    SELECT
        t.station,
        t.date,
        t.hour,
        t.is_workday,
        t.entries,
        w1.entries AS last_week,
        (coalesce(w1.entries, 0) + coalesce(w2.entries, 0) + coalesce(w3.entries, 0) + coalesce(w4.entries, 0))
            / nullif(
                (w1.entries IS NOT NULL)::INTEGER + (w2.entries IS NOT NULL)::INTEGER
                + (w3.entries IS NOT NULL)::INTEGER + (w4.entries IS NOT NULL)::INTEGER,
                0
            ) AS mean_4_weeks,
        w52.entries AS last_year
    FROM normal_hours AS t
    LEFT JOIN normal_hours AS w1  ON w1.station = t.station  AND w1.hour = t.hour  AND w1.date = t.date - 7
    LEFT JOIN normal_hours AS w2  ON w2.station = t.station  AND w2.hour = t.hour  AND w2.date = t.date - 14
    LEFT JOIN normal_hours AS w3  ON w3.station = t.station  AND w3.hour = t.hour  AND w3.date = t.date - 21
    LEFT JOIN normal_hours AS w4  ON w4.station = t.station  AND w4.hour = t.hour  AND w4.date = t.date - 28
    LEFT JOIN normal_hours AS w52 ON w52.station = t.station AND w52.hour = t.hour AND w52.date = t.date - 364
    WHERE t.date BETWEEN DATE '2022-07-01' AND DATE '2022-12-31'
"""

with stage("03.baselines"):
    con.execute(baseline_query)

all_rows, complete_rows = fetch_row(con, """
    SELECT
        count(*),
        count(*) FILTER (WHERE last_week IS NOT NULL AND mean_4_weeks IS NOT NULL AND last_year IS NOT NULL)
    FROM baseline_eval
""")
print(f"station-hours in Jul-Dec 2022: {all_rows:,}")
print(f"rows where all 3 baselines exist: {complete_rows:,} ({complete_rows / all_rows:.1%})")

[perf] 03.baselines: 0.20s | 13.82 cores | peak RAM 0.769 GB
station-hours in Jul-Dec 2022: 220,800
rows where all 3 baselines exist: 220,800 (100.0%)


In [47]:
segments = {
    "all hours": "true",
    "workday": "is_workday",
    "weekend / holiday": "NOT is_workday",
    "AM peak (workday 6-9)": "is_workday AND hour BETWEEN 6 AND 9",
    "PM peak (workday 15-18)": "is_workday AND hour BETWEEN 15 AND 18",
}
baselines = {
    "last week": "last_week",
    "mean of 4 weeks": "mean_4_weeks",
    "last year": "last_year",
}

wape_rows = []
for segment, condition in segments.items():
    for baseline, column in baselines.items():
        wape, rows = fetch_row(con, f"""
            SELECT 100 * sum(abs(entries - {column})) / sum(entries), count(*)
            FROM baseline_eval
            WHERE last_week IS NOT NULL
              AND mean_4_weeks IS NOT NULL
              AND last_year IS NOT NULL
              AND {condition}
        """)
        wape_rows.append({"segment": segment, "baseline": baseline, "wape_pct": wape, "rows": rows})

wape_table = pd.DataFrame(wape_rows)
wape_table.pivot(index="segment", columns="baseline", values="wape_pct").loc[list(segments)].round(1)

baseline,last week,last year,mean of 4 weeks
segment,,,
all hours,20.2,33.7,17.5
workday,16.2,33.1,13.9
weekend / holiday,37.2,36.0,33.1
AM peak (workday 6-9),15.2,33.8,13.1
PM peak (workday 15-18),13.7,31.6,11.8


In [48]:
baseline_colors = {"last week": viz.ACCENT, "mean of 4 weeks": viz.SERIES[2], "last year": viz.CONTEXT}

fig = go.Figure()
for baseline, color in baseline_colors.items():
    rows = wape_table[wape_table["baseline"] == baseline]
    fig.add_trace(go.Bar(
        name=baseline,
        x=rows["segment"],
        y=rows["wape_pct"],
        marker_color=color,
        hovertemplate="%{x}: WAPE %{y:.1f}%<extra>%{fullData.name}</extra>",
    ))

fig.update_layout(barmode="group", hovermode="closest")
fig.update_yaxes(title_text="WAPE (lower is better)", ticksuffix="%")

best_baseline = None
best_wape = None
for row in wape_rows:
    if row["segment"] != "all hours":
        continue
    if best_wape is None or row["wape_pct"] < best_wape:
        best_baseline = row["baseline"]
        best_wape = row["wape_pct"]

title = f"The bar for M1: {best_baseline} already reaches WAPE {best_wape:.1f}%"
subtitle = "Naive forecasts of station-hour entries, Jul-Dec 2022 (M1 validation period), normal days, same rows for all three"
viz.titled(fig, title, subtitle)
viz.save(fig, "03_baseline_wape")
fig

### ข้อสังเกต (ส่วนที่ 5)

- ACF ของทั้งระบบปี 2019 มี peak ทุก 24 ชั่วโมง แต่ peak ที่ 168 ชั่วโมง (r = 0.95) สูงกว่า 24 ชั่วโมง (r = 0.79) มาก และ 336 ชั่วโมงยังสูงอยู่ (0.94) ค่าที่ 144 และ 192 ชั่วโมงต่ำกว่า เพราะเป็นคนละวันในสัปดาห์ วันเดียวกันในสัปดาห์จึงเป็นตัวบอกอนาคตที่ดีที่สุด ซึ่งพอดีกับเงื่อนไขของ M1 ที่ห้ามใช้ lag สั้นกว่า 168 ชั่วโมง
- ระดับสถานี-ชั่วโมง correlation กับ 1-4 สัปดาห์ก่อนอยู่ที่ ~0.93-0.95 ทั้งปี 2019 และปี 2022 ส่วนค่าเมื่อ 52 สัปดาห์ก่อนลดลงมากที่สุด จาก 0.969 เหลือ 0.943
- baseline ในช่วง valid ของ M1 (ก.ค.-ธ.ค. 2022) ค่าเฉลี่ย 4 สัปดาห์ดีที่สุด WAPE 17.5% รองลงมาคือสัปดาห์ก่อน 20.2% ส่วนปีก่อนแย่ที่สุด 33.7%
- ค่าเมื่อปีก่อนยังมี correlation สูง (0.94) แต่ WAPE แย่ แปลว่ารูปทรงยังเหมือนเดิม แต่ระดับเปลี่ยนไปเยอะในหนึ่งปี เพราะผู้โดยสารกำลังฟื้นตัว (ดู trend ในหัวข้อ 2.1) lag ปีก่อนจึงต้องใช้คู่กับ feature ที่บอกระดับล่าสุด
- วันทำงานพยากรณ์ง่ายกว่ามาก (WAPE 13.9%) ส่วนวันหยุดยากที่สุด (33.1%) เพราะคนน้อยกว่าและผันผวนตามกิจกรรม รวมถึงวันหยุดราชการที่สัปดาห์ก่อนเป็นวันทำงานปกติ
- ช่วง peak พยากรณ์ง่ายกว่าค่าเฉลี่ยทั้งวัน (AM 13.1%, PM 11.8%) เพราะเป็นการเดินทางไปทำงานที่สม่ำเสมอ

ใช้ต่อที่ไหน
- M1: lag feature หลักคือค่าเมื่อ 1, 2, 3, 4 สัปดาห์ก่อน พร้อมค่าเฉลี่ย/มัธยฐาน/ส่วนเบี่ยงเบนของ 4 สัปดาห์ ใส่ lag 52 สัปดาห์ด้วย แต่ต้องมี feature ระดับล่าสุด (ค่าเฉลี่ย 7 และ 28 วัน) และ trend ช่วยปรับระดับ
- M1: baseline ที่ต้องชนะคือค่าเฉลี่ย 4 สัปดาห์ WAPE 17.5% (และสัปดาห์ก่อน 20.2%) และต้องรายงานแยก segment เพราะวันหยุดยากกว่าวันทำงานมาก
- M1: feature วันหยุดสำคัญมาก เพราะ baseline พลาดหนักที่สุดในวันหยุด
- M3: WAPE ของ baseline บอกระดับความคลาดเคลื่อนปกติ ใช้ตั้งเกณฑ์ว่าห่างแค่ไหนถึงนับเป็นเหตุการณ์ผิดปกติ

## 6. สรุป EDA

### ข้อมูลและการกระจาย
1. ระดับชั่วโมงข้อมูลเบาบาง แถว OD-ชั่วโมง 27% มีแค่ 1 ทริป และมีชั่วโมงที่เป็นศูนย์ ~14% (ส่วนใหญ่ช่วงตี 1-4)
   → M1 ใช้ WAPE เป็นตัวชี้วัดหลักแทน MAPE
2. variance โตเร็วกว่า mean (overdispersion, p ≈ 1.4)
   → M1 ใช้ objective Tweedie ให้ Optuna จูน power ในช่วง 1.1-1.9

### เวลา
3. ผู้โดยสารวันทำงานต้นปี 2023 อยู่ที่ 37% ของปี 2019 ส่วนวันหยุดอยู่ที่ 51%
   → Q1 ต้องแยกวันทำงานกับวันหยุด
4. หลัง COVID จันทร์/ศุกร์เบากว่าอังคาร-พฤหัสชัดเจน (85/83 เทียบกับ 100)
   → Q1 (hybrid work) และ M1 ต้องมี feature วันในสัปดาห์
5. วันทำงานยังเป็น double peak แต่ peak เช้าเล็กลง (33.7% เหลือ 30.6% ของวัน) และกลางวันหนาแน่นขึ้น
   → Q2
6. ฤดูกาลรายเดือนมีน้อย (±5%) ยกเว้นธันวาคม-มกราคม
   → M1 ให้น้ำหนัก feature เดือนน้อย
7. วันพีคผิดปกติส่วนใหญ่เกิดซ้ำทุกปี (SF Pride ~2 เท่า, ขบวนพาเหรดแชมป์)
   → M1 ใส่ flag วัน Pride และ M3 ใช้เป็นตัวอย่างที่รู้คำตอบอยู่แล้ว

### สถานีและ OD
8. downtown SF เสียผู้โดยสารมากที่สุด (~190,000 คนต่อวัน หรือ ~37% ของที่หายไปทั้งระบบ) ส่วน Dublin line ฟื้นช้าที่สุดในแง่ % (32%)
   → Q1
9. ตอนเช้ามีแค่ 10 สถานีที่เป็นปลายทาง และ profile รายชั่วโมงแยกประเภทสถานีได้ชัด
   → M2 (feature ของ clustering)
10. WARM เปลี่ยนจากต้นทางเป็นปลายทางตอนเช้า (26% เป็น 52%) ยังไม่ทราบสาเหตุ
    → Q4
11. 8% ของคู่ OD รวมกันได้ครึ่งหนึ่งของทริป และทริปข้าม Transbay Tube ยังเป็น ~52% ของวันทำงาน (~79,000 ทริปต่อวัน) สองทิศเท่ากันทั้งวัน
    → Q3 ต้องดูแยกชั่วโมงและทิศ

### สำหรับโมเดล
12. ค่าที่ชั่วโมงเดียวกันเมื่อสัปดาห์ก่อน (r = 0.95) ดีกว่าเมื่อวาน (0.79)
    → M1 ใช้ lag 1-4 สัปดาห์และสถิติของ 4 สัปดาห์
13. ค่าเมื่อปีก่อนรูปทรงเหมือนเดิม แต่ระดับเปลี่ยนไปมาก (WAPE 33.7%)
    → M1 ต้องมี feature ระดับล่าสุดและ trend คู่กัน
14. baseline ที่ M1 ต้องชนะในช่วง valid: ค่าเฉลี่ย 4 สัปดาห์ WAPE 17.5% (วันทำงาน 13.9%, วันหยุด 33.1%)
    → M1 รายงานแยก segment และ M3 ใช้ตั้งเกณฑ์ความผิดปกติ

ต่อไป: 04_q1_recovery